# 5·6장 통합 과제 — 신규 셔틀의 GTFS와 경로 효과

5장에서 시간표 여섯 행을 만들고, 6장에서 환승과 라운드 갱신을 학습한 뒤 진행합니다.
1단계에서 GTFS를 생성·저장하고, 2단계에서 두 출발시각을 비교합니다.
3단계에서는 24개 출발시각과 배차간격 10분 안으로 넓힙니다.

기본안은 기존 정류장 네 곳을 지나는 가상 단방향 하남 셔틀입니다.
구간시간은 설계 가정이며, 실제 도로 주행시간이나 차량 운영비를 추정하지 않습니다.
실행 전에 [과제 안내](README.md)의 사전 질문과 제출물을 읽습니다.

아래 셀에서 패키지 경로와 표·그림 도구를 준비합니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import pandas as pd
import matplotlib.pyplot as plt
from lab import expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

필요한 하남 자료와 시간·경로 함수를 읽습니다. 이 노트북만 새 커널에서 실행해도 됩니다.

In [ ]:
from smartmob.data import load_gtfs, parse_gtfs_time, seconds_to_gtfs_time
from smartmob.teaching.raptor import (
    INF, TransitData, raptor as ref_raptor, journey, summarize,
)

feed = load_gtfs("hanam")
hanam = feed

def clock_label(seconds):
    return "미도달" if seconds == INF else seconds_to_gtfs_time(seconds)

## 1. 노선 설계와 GTFS 생성 ★★

하남시청에서 하남문화예술회관·미사강변브라운스톤을 거쳐 미사역으로 가는 가상 셔틀을 만듭니다.
기존 정류장의 위치와 식별자는 재사용하고, 노선·운행·시간표는 직접 작성합니다.
아래 기본안부터 실행한 뒤 정류장 순서, 배차간격, 구간시간 중 하나 이상을 바꾸어 자기 안을 만듭니다.
노선 개설이나 실제 버스 운행을 뜻하지 않는 수업용 시간표입니다.

산출물은 GTFS 텍스트 표 6개를 담은 ZIP, 노선도, 설계 조건표입니다.
이 과제 2단계에서는 이 ZIP을 기존 시간표에 추가해 통행시간과 환승 횟수를 비교합니다.
원래 `feed`와 `data/hanam/gtfs`는 그대로 두고 `outputs/`에 저장합니다.

### 1.1 정류장 순서와 운행 조건 정하기

정류장 이름을 검색해 지도와 방향을 확인한 뒤 아래 목록을 고칩니다.
`run_minutes`는 이웃한 두 정류장 사이의 차내시간입니다. 정류장 4개이면 값은 3개입니다.
이 기본안은 한 방향만 다니며 정차시간은 0초로 둡니다.

In [ ]:
scenario_name = "my_shuttle"
new_route_id = "LAB_SHUTTLE_01"
new_route_name = "실습셔틀"
service_date = "20260923"
stop_order = [
    "BS_TAGO_GGB227000034",  # 하남시청
    "BS_TAGO_GGB227000224",  # 하남문화예술회관
    "BS_TAGO_GGB227000506",  # 미사강변브라운스톤
    "BS_TAGO_GGB227000658",  # 미사역
]
run_minutes = [4, 8, 2]
first_departure = "07:05:00"
service_end = "09:00:00"
headway_min = 20

첫 출발은 07:05이며 09:00 미만까지 20분마다 출발합니다. 기본안은 6편입니다.
구간시간은 관측값이 아니라 설계 가정입니다. 배차를 줄이는 안과 구간시간을 줄이는 안을 구분합니다.
설계 조건이 서로 맞는지 검사한 뒤 정류장 표를 만듭니다.

In [ ]:
from datetime import datetime

assert scenario_name and Path(scenario_name).name == scenario_name
assert scenario_name not in {".", ".."}
assert len(stop_order) >= 2 and len(set(stop_order)) == len(stop_order)
assert len(run_minutes) == len(stop_order) - 1
assert all(isinstance(m, int) and m > 0 for m in run_minutes)
assert isinstance(headway_min, int) and headway_min > 0
assert parse_gtfs_time(first_departure) < parse_gtfs_time(service_end)
assert new_route_id not in set(feed["routes"]["route_id"])
datetime.strptime(service_date, "%Y%m%d")
new_stops = feed["stops"].set_index("stop_id").loc[stop_order].reset_index().copy()
new_stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

선택한 순서대로 정류장이 나옵니다. 알 수 없는 `stop_id`를 넣으면 이 단계에서 멈춥니다.
정류장을 바꾸었다면 지도와 구간시간도 함께 확인합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(new_stops["stop_lon"], new_stops["stop_lat"], "o-", color="#2869a4")
for i, row in enumerate(new_stops.itertuples(), start=1):
    label_left = row.stop_lon > new_stops["stop_lon"].median()
    ax.annotate(f"{i}. {row.stop_name}", (row.stop_lon, row.stop_lat),
                xytext=(-8 if label_left else 8, 8), textcoords="offset points",
                ha="right" if label_left else "left")
ax.set(xlabel="경도", ylabel="위도", title=f"{new_route_name} · 가상 정류장 방문 순서")
ax.set_aspect(1 / 0.79)
ax.margins(0.25)
fig.tight_layout()
new_route_figure = fig

선은 정류장 방문 순서이며 실제 도로를 따라가는 경로가 아닙니다.
이 그림으로 순서가 크게 돌아가는지 확인합니다. 도로 통행 가능 여부와 실제 주행시간은 별도의 문제입니다.

### 1.2 운영기관·노선·운행일 작성하기

단독 GTFS에는 앞서 읽은 다섯 표에 `agency.txt`를 추가합니다.
기관 이름과 URL은 가상이며 시간대는 `Asia/Seoul`입니다.
새 ZIP의 버스 코드는 [GTFS 명세](https://gtfs.org/documentation/schedule/reference/)에 따라 `3`으로 씁니다.
교재의 하남 파일에서 시내버스를 `0`으로 읽는 것과 구분합니다.

In [ ]:
new_agency_id = "LAB_AGENCY"
new_service_id = "LAB_SERVICE_" + service_date
new_agency = pd.DataFrame([{
    "agency_id": new_agency_id, "agency_name": "수업용 가상 셔틀",
    "agency_url": "https://example.org/teaching-shuttle", "agency_timezone": "Asia/Seoul",
}])
new_routes = pd.DataFrame([{
    "route_id": new_route_id, "agency_id": new_agency_id,
    "route_short_name": new_route_name, "route_long_name": "수업용 가상 단방향 셔틀",
    "route_type": 3,
}])
weekdays = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
new_calendar = pd.DataFrame([{
    "service_id": new_service_id, **{day: 1 for day in weekdays},
    "start_date": service_date, "end_date": service_date,
}])
display(new_routes, new_calendar)

`route_id` 하나와 `service_id` 하나를 만들었습니다. 적용 시작일과 종료일이 같으므로 지정한 하루만 운행합니다.
요일 값이 모두 1이어도 적용 기간 밖의 날에는 운행하지 않습니다.

### 1.3 출발편마다 trip_id 만들기

07:05, 07:25처럼 첫 정류장 출발시각을 만든 뒤 각 출발편에 고유 식별자를 붙입니다.
노선 번호는 같아도 `trip_id`는 달라야 합니다.

In [ ]:
first_s = parse_gtfs_time(first_departure)
end_s = parse_gtfs_time(service_end)
new_departures = list(range(first_s, end_s, headway_min * 60))
new_trip_ids = [f"{new_route_id}_{i:03d}" for i in range(len(new_departures))]
new_trips = pd.DataFrame({
    "route_id": new_route_id, "service_id": new_service_id, "trip_id": new_trip_ids,
    "direction_id": 0,
})
print("운행 수:", len(new_trips))
print("첫 정류장 출발:", [seconds_to_gtfs_time(t) for t in new_departures])

기본안은 07:05부터 08:45까지 6편입니다. 09:00은 출발시각 상한이며 포함하지 않습니다.
배차간격을 10분으로 바꾸면 같은 시간 범위에서 12편이 됩니다.

### 1.4 정류장마다 도착·출발시각 채우기

기본안의 누적 통행시간은 `[0, 4, 12, 14]`분입니다.
첫 정류장 출발시각에 누적 시간을 더하면 각 정류장의 시각이 됩니다.
각 `trip_id`에 정류장 수만큼 행을 만들고 `stop_sequence`는 1부터 증가시킵니다.

In [ ]:
from itertools import accumulate

offsets_s = [0, *accumulate(m * 60 for m in run_minutes)]
new_rows = []
for trip_id, departure_s in zip(new_trip_ids, new_departures):
    for sequence, (stop_id, offset) in enumerate(zip(stop_order, offsets_s), start=1):
        at = seconds_to_gtfs_time(departure_s + offset)
        new_rows.append((trip_id, at, at, stop_id, sequence))
new_stop_times = pd.DataFrame(new_rows, columns=[
    "trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence",
])
new_stop_times.head(len(stop_order))

첫 편은 07:05→07:09→07:17→07:19입니다. 전체 방문 기록은 6편×4곳=24행입니다.
`seconds_to_gtfs_time`을 쓰므로 자정 이후에도 24시 이상의 시각이 유지됩니다.
왕복 노선을 만들려면 반대 방향 정류장을 따로 고르고 별도의 운행과 방문 순서를 추가합니다.
단방향 시간표의 순서만 뒤집어 실제 승차 방향까지 확인한 것으로 보지는 않습니다.

### 1.5 식별자와 시간 순서 검사하기

`validate_feed`는 필수 표·컬럼만 확인합니다. 아래에서 식별자 연결, 방문 순서, 시각도 검사합니다.
정류장·운행의 이름을 바꾸다가 연결이 끊기면 저장 전에 발견할 수 있습니다.

In [ ]:
from smartmob.data import validate_feed, save_gtfs, load_gtfs_feed

new_feed = {"agency": new_agency, "stops": new_stops, "routes": new_routes,
            "trips": new_trips, "stop_times": new_stop_times, "calendar": new_calendar}
validate_feed(new_feed)
for table, key in [("agency", "agency_id"), ("stops", "stop_id"), ("routes", "route_id"),
                   ("trips", "trip_id"), ("calendar", "service_id")]:
    assert new_feed[table][key].is_unique
for left, right, key in [("routes", "agency", "agency_id"), ("trips", "routes", "route_id"),
                         ("trips", "calendar", "service_id"), ("stop_times", "trips", "trip_id"),
                         ("stop_times", "stops", "stop_id")]:
    assert set(new_feed[left][key]) <= set(new_feed[right][key])
assert set(new_stop_times["trip_id"]) == set(new_trips["trip_id"])
assert not new_stop_times.duplicated(["trip_id", "stop_sequence"]).any()
print("표와 식별자의 연결을 확인했습니다.")

모든 운행이 노선·운행일·정류장 표에 연결됩니다. 다음으로 운행 안에서 시간이 뒤로 가지 않는지 봅니다.
도착은 출발보다 늦을 수 없고 다음 정류장 도착은 이전 정류장 출발보다 이를 수 없습니다.

In [ ]:
for trip_id, group in new_stop_times.groupby("trip_id"):
    group = group.sort_values("stop_sequence")
    assert group["stop_sequence"].tolist() == list(range(1, len(stop_order) + 1))
    assert group["stop_id"].tolist() == stop_order
    arr = group["arrival_time"].map(parse_gtfs_time).tolist()
    dep = group["departure_time"].map(parse_gtfs_time).tolist()
    assert all(a <= d for a, d in zip(arr, dep))
    assert all(d <= a for d, a in zip(dep[:-1], arr[1:]))
print("운행별 방문 순서와 시각을 확인했습니다.")

이 검사는 이번 실습의 표 관계와 시간 조건을 확인합니다. GTFS 명세 전체의 검증을 대신하지는 않습니다.
기본안에서는 정차시간을 0초로 정했으므로 각 정류장의 도착·출발시각이 같습니다.

### 1.6 ZIP으로 저장하고 다시 읽기

표를 UTF-8 CSV인 `.txt`로 저장하고 파일 6개를 ZIP의 최상위에 넣습니다.
`scenario_name`을 바꾸면 다른 설계안을 별도 폴더에 보관할 수 있습니다.
같은 이름으로 다시 실행하면 그 안의 실습 산출물을 갱신합니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

scenario_dir = ROOT / "outputs" / scenario_name
new_feed_dir = scenario_dir / "gtfs"
save_gtfs(new_feed, new_feed_dir)
new_zip_path = scenario_dir / "gtfs.zip"
with ZipFile(new_zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in new_feed:
        archive.write(new_feed_dir / f"{name}.txt", arcname=f"{name}.txt")
new_route_figure.savefig(scenario_dir / "route.png", dpi=150, bbox_inches="tight")
print(new_zip_path)

기본 저장 위치는 `outputs/my_shuttle/gtfs.zip`입니다. 노선도는 같은 폴더의 `route.png`입니다.
다시 읽어 ZIP 내용과 행 수를 확인합니다. `load_gtfs_feed`는 탐색용 다섯 표를 읽으므로
운영기관 표는 ZIP에서 별도로 확인합니다.

In [ ]:
with ZipFile(new_zip_path) as archive:
    assert set(archive.namelist()) == {f"{name}.txt" for name in new_feed}
    agency_back = pd.read_csv(archive.open("agency.txt"), dtype=str)
    assert agency_back.iloc[0]["agency_timezone"] == "Asia/Seoul"
new_loaded = load_gtfs_feed(new_zip_path)
assert len(new_loaded["trips"]) == len(new_departures)
assert len(new_loaded["stop_times"]) == len(new_departures) * len(stop_order)
assert set(new_loaded["routes"]["route_type"].astype(int)) == {3}
pd.DataFrame({"행 수": {name: len(table) for name, table in new_loaded.items()}})

기본안의 표는 정류장 4행, 노선 1행, 운행 6행, 방문 24행, 운행일 1행입니다.
ZIP 안에는 여기에 운영기관 1행이 더 있습니다. 설계 조건도 함께 남깁니다.

In [ ]:
new_design = pd.DataFrame([{
    "노선": new_route_name, "운행일": service_date, "정류장 수": len(stop_order),
    "첫 출발": first_departure, "출발 종료(미만)": service_end,
    "배차간격(분)": headway_min, "편도시간(분)": sum(run_minutes),
    "운행 수": len(new_trips), "방문 행 수": len(new_stop_times),
    "정류장 순서": " → ".join(new_stops["stop_name"]),
}])
new_design.to_csv(scenario_dir / "design.csv", index=False, encoding="utf-8-sig")
new_design

### 직접 바꾸어 제출하기

기본안을 실행한 뒤 정류장 순서·배차간격·구간시간 중 하나 이상을 바꾸고 1.1절부터 다시 실행해 봅시다.
산출물은 `gtfs.zip`, `route.png`, `design.csv`, 설계 이유 3문장입니다.
10분 간격안은 `scenario_name = "my_shuttle_10min"`으로 저장해 20분 간격안과 함께 보관합니다.

이 과제 2단계에서 어떤 승객의 시간이 줄었는지 확인합니다. 배차를 촘촘하게 하면 운행 편수도 늘어납니다.
이 실습은 차량 회차·운전기사 근무·운영비를 계산하지 않으므로 경로 개선만으로 운영안을 확정하지 않습니다.
추가 연습 ★★★: 반대 방향의 정류장과 운행을 추가해 패턴 두 개를 만들고 ZIP을 다시 검사해 봅시다.

## 2. 두 출발시각의 경로부터 비교하기 ★★

이 과제 1단계에서 저장한 `outputs/my_shuttle/gtfs.zip`을 읽습니다.
새 노선을 추가하기 전과 후에 같은 운행일·출발 정류장·도착 정류장·탑승 상한을 사용합니다.
새 파일이 아직 없으면 이 절만 건너뜁니다. 1단계를 마친 뒤 아래 셀부터 다시 실행합니다.

이 비교는 하남시청 정류장에 준비된 시각부터 미사강변브라운스톤 정류장까지입니다.
6장 기본 실습의 건물 좌표에서 출발하는 22.9분과는 출발 조건이 다릅니다.
먼저 어느 출발시각에서 셔틀을 탈 수 있을지 예상해 봅시다.

In [ ]:
from smartmob.data import load_gtfs_feed

my_zip_path = ROOT / "outputs/my_shuttle/gtfs.zip"
added_feed = load_gtfs_feed(my_zip_path) if my_zip_path.exists() else None
if added_feed is None:
    print("이 과제 1단계에서 GTFS ZIP을 만든 뒤 2단계를 실행합니다.")
else:
    print("읽은 파일:", my_zip_path)
    display(added_feed["routes"], added_feed["calendar"])

ZIP의 버스 코드는 표준의 `3`입니다. 복사본을 하남 실습 코드인 `0`으로 바꾼 뒤 합칩니다.
저장한 ZIP은 그대로 둡니다. `route_id`, `trip_id`, `service_id`가 기존 값과 겹치면 합치지 않습니다.

In [ ]:
scenario_feed = None
if added_feed is not None:
    assert set(added_feed["routes"]["route_type"].astype(int)) == {3}, "버스 노선 실습입니다."
    added_local = {name: table.copy() for name, table in added_feed.items()}
    added_local["routes"]["route_type"] = 0
    scenario_feed = {name: table.copy() for name, table in hanam.items()}
    for table, key in [("routes", "route_id"), ("trips", "trip_id"), ("calendar", "service_id")]:
        assert not set(hanam[table][key]) & set(added_local[table][key]), f"{key} 중복"
        scenario_feed[table] = pd.concat([hanam[table], added_local[table]], ignore_index=True)
    scenario_feed["stop_times"] = pd.concat(
        [hanam["stop_times"], added_local["stop_times"]], ignore_index=True,
    )

노선과 시간표만 추가했습니다. 이번 실습은 기존 정류장 위치를 재사용하므로 정류장 표를 중복해서 붙이지 않습니다.
같은 `stop_id`인데 좌표나 이름을 바꿨다면 먼저 설계를 확인합니다.

In [ ]:
if scenario_feed is not None:
    stop_reference = hanam["stops"].set_index("stop_id")
    assert set(added_local["stops"]["stop_id"]) <= set(stop_reference.index)
    for stop in added_local["stops"].itertuples():
        original = stop_reference.loc[stop.stop_id]
        assert original.stop_name == stop.stop_name
        assert abs(float(original.stop_lat) - float(stop.stop_lat)) < 1e-6
        assert abs(float(original.stop_lon) - float(stop.stop_lon)) < 1e-6
    print("기존 정류장의 이름·좌표가 유지됩니다.")

표를 합칠 때 이름만 같다고 같은 정류장으로 보지 않습니다.
이 실습은 1단계에서 고른 기존 `stop_id`를 유지하는 범위입니다.

### 운행 날짜를 먼저 선택하기

`raptor`에 날짜 인수가 없으므로 시간표를 준비하기 전에 운행일을 고릅니다.
아래 함수는 이번 실습의 `calendar` 적용 기간과 요일을 사용합니다.
다른 자료의 `calendar_dates` 예외나 전날에서 이어지는 심야 운행은 이 함수에 포함하지 않습니다.

In [ ]:
from datetime import datetime

def feed_on_date(source, date_text):
    day_names = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
    day = day_names[datetime.strptime(date_text, "%Y%m%d").weekday()]
    calendar = source["calendar"].astype(str)
    active = calendar.loc[(calendar["start_date"] <= date_text)
                          & (calendar["end_date"] >= date_text) & (calendar[day] == "1"), "service_id"]
    selected = {name: table.copy() for name, table in source.items()}
    selected["trips"] = source["trips"][source["trips"]["service_id"].isin(active)]
    selected["stop_times"] = source["stop_times"][
        source["stop_times"]["trip_id"].isin(selected["trips"]["trip_id"])
    ]
    return selected

이 함수는 운행과 정류장 방문을 함께 거릅니다. 정류장 목록은 비교 전후에 그대로 유지합니다.
새 노선의 적용 시작일을 비교 날짜로 정하고, 그날 실제로 선택된 운행 수를 확인합니다.

In [ ]:
if scenario_feed is not None:
    query_date = str(added_local["calendar"].iloc[0]["start_date"])
    baseline_day = feed_on_date(hanam, query_date)
    scenario_day = feed_on_date(scenario_feed, query_date)
    active_new = set(scenario_day["trips"]["trip_id"]) & set(added_local["trips"]["trip_id"])
    assert active_new, "비교 날짜에 신규 운행이 없습니다. calendar를 확인합니다."
    baseline_data = TransitData.from_gtfs(baseline_day)
    scenario_data = TransitData.from_gtfs(scenario_day)
    print("비교 날짜:", query_date, "신규 운행:", len(active_new), "편")

기본안에서는 2026-09-23의 신규 운행 6편이 선택됩니다.
1단계에서 배차간격이나 운행 날짜를 바꾸면 실제 선택된 수를 읽습니다.
동일한 정류장에서 같은 시각에 출발하는 요청들을 두 시간표에 넣습니다.

In [ ]:
comparison_rows = []
compare_origin = "BS_TAGO_GGB227000034"
compare_target = "BS_TAGO_GGB227000506"
if scenario_feed is not None:
    for ready_s in [8 * 3600, 8 * 3600 + 10 * 60]:
        for label, transit_data in [("기존", baseline_data), ("신규 추가", scenario_data)]:
            source_idx = transit_data.index_of[compare_origin]
            target_idx = transit_data.index_of[compare_target]
            answer = ref_raptor(transit_data, [(source_idx, 0)], ready_s, max_rounds=5)
            parts = journey(transit_data, answer, target_idx)
            info = summarize(transit_data, parts, ready_s)
            comparison_rows.append({
                "출발": seconds_to_gtfs_time(ready_s), "시나리오": label,
                "도착": clock_label(answer.best[target_idx]),
                "통행시간(분)": info.get("total_min"), "환승 수": info.get("transfers"),
                "이용 노선": " → ".join(info.get("routes", [])),
            })

먼저 08:00과 08:10 두 요청만 비교합니다. 기본안은 08:05에 출발하므로 누가 탈 수 있는지 예상합니다. 미도달은 빈값으로 남기고 0분으로 바꾸지 않습니다.
신규 셔틀이 있어도 기존 경로가 빠르면 기존 경로를 선택할 수 있습니다.

In [ ]:
if comparison_rows:
    new_comparison = pd.DataFrame(comparison_rows)
    time_comparison = new_comparison.pivot(index="출발", columns="시나리오", values="통행시간(분)")
    time_comparison["단축(분)"] = time_comparison["기존"] - time_comparison["신규 추가"]
    display(time_comparison)
    display(new_comparison[new_comparison["출발"].isin(["08:00:00", "08:10:00"])])

기본안의 08:00 요청은 22.9분에서 17분으로 줄고, 08:10 요청은 18.6분을 유지합니다. 이용 노선과 환승 수를 함께 읽습니다.
음수라면 입력 조건과 제공된 RAPTOR의 간소화 영향을 확인합니다.
이 값은 시간표 기반 계산이며 실제 승객 수를 가중한 평균이 아닙니다.
그래프와 표를 신규 노선 ZIP 옆에 저장합니다.

## 3. 출발시각과 배차간격으로 넓히기 ★★

앞의 두 요청을 설명한 뒤 07:00~08:55를 5분 간격으로 비교합니다.
24개 요청에서도 시각별로 같은 출발·도착 정류장과 탑승 상한을 유지합니다.
기다림 때문에 셔틀 효과가 주기적으로 달라질지 먼저 예상해 봅니다.
아래 코드는 2단계의 질의를 반복하며 비교표를 새로 만듭니다.

In [ ]:
comparison_rows = []
compare_origin = "BS_TAGO_GGB227000034"
compare_target = "BS_TAGO_GGB227000506"
if scenario_feed is not None:
    for ready_s in range(7 * 3600, 9 * 3600, 5 * 60):
        for label, transit_data in [("기존", baseline_data), ("신규 추가", scenario_data)]:
            source_idx = transit_data.index_of[compare_origin]
            target_idx = transit_data.index_of[compare_target]
            answer = ref_raptor(transit_data, [(source_idx, 0)], ready_s, max_rounds=5)
            parts = journey(transit_data, answer, target_idx)
            info = summarize(transit_data, parts, ready_s)
            comparison_rows.append({
                "출발": seconds_to_gtfs_time(ready_s), "시나리오": label,
                "도착": clock_label(answer.best[target_idx]),
                "통행시간(분)": info.get("total_min"), "환승 수": info.get("transfers"),
                "이용 노선": " → ".join(info.get("routes", [])),
            })

기존·신규 추가 각각 24행으로 총 48행입니다. 같은 출발시각끼리 나란히 놓아 단축시간을 읽습니다.

In [ ]:
if comparison_rows:
    new_comparison = pd.DataFrame(comparison_rows)
    time_comparison = new_comparison.pivot(index="출발", columns="시나리오", values="통행시간(분)")
    time_comparison["단축(분)"] = time_comparison["기존"] - time_comparison["신규 추가"]
    display(time_comparison)
    display(new_comparison[new_comparison["출발"].isin(["08:00:00", "08:10:00"])])

통행시간 감소가 큰 시각과 0인 시각을 고릅니다. 아래 그림을 저장한 뒤 두 경로의 이용 노선을 함께 설명합니다.

In [ ]:
if comparison_rows:
    ax = time_comparison[["기존", "신규 추가"]].plot(figsize=(10, 4), marker="o")
    ax.set(xlabel="출발 정류장에 준비된 시각", ylabel="통행시간(분)", title="신규 노선 추가 전후")
    ax.figure.tight_layout()
    ax.figure.savefig(my_zip_path.parent / "comparison.png", dpi=150, bbox_inches="tight")
    new_comparison.to_csv(my_zip_path.parent / "comparison.csv", index=False, encoding="utf-8-sig")
    print("저장 위치:", my_zip_path.parent)

출발시각에 따라 단축 폭이 달라지는 이유를 배차간격과 탑승한 운행으로 설명해 봅시다.
산출물은 `comparison.csv`, `comparison.png`, 개선된 시각과 개선되지 않은 시각의 경로 설명입니다.

추가 비교 ★★: 1단계에서 10분 간격안을 다른 폴더에 저장한 뒤 `my_zip_path`를 바꾸어 이 절을 다시 실행합니다.
기존·20분 간격·10분 간격의 통행시간과 추가 운행 편수를 함께 적습니다.
각 안은 원래 `hanam`에 한 번만 추가합니다. 이전 안에 새 안을 누적해서 합치지 않습니다.